In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os, sys
os.environ["USER"] = "kaggle"   # deterministic prediction output paths

# Repo to clone. Upstream royerlab is the stock baseline. To use OUR changes
# (PU detection loss, etc.), push them to your fork and point REPO_URL at it:
#   REPO_URL = "https://github.com/<your-github-username>/kaggle-cell-tracking-competition.git"
REPO_URL = "https://github.com/Caleb-Kelly-25/kaggle-cell-tracking-competition.git"

# Clone once. NOTE: if /kaggle/working/repo already exists this is skipped --
# start a fresh session (or delete it) to switch REPO_URL.
os.system(f"test -d /kaggle/working/repo || git clone -q {REPO_URL} /kaggle/working/repo")
%cd /kaggle/working/repo

# Make `tracking_cellmot` importable in THIS kernel AND in the !python subprocesses below.
# (pip install -e . registers its import hook only at interpreter startup, so it does NOT
# take effect in an already-running kernel -- that's the ModuleNotFoundError you hit.)
SRC = "/kaggle/working/repo/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")

# Pin numpy + scipy to Kaggle's preinstalled versions so nothing below can upgrade them
# under the running kernel -- that mismatch is what broke `import numpy` last time.
import numpy, scipy
with open("/kaggle/working/constraints.txt", "w") as f:
    f.write(f"numpy=={numpy.__version__}\nscipy=={scipy.__version__}\n")

!pip install -q -c /kaggle/working/constraints.txt \
    "tracksdata @ git+https://github.com/royerlab/tracksdata@main" \
    "zarr>=3.0.10" geff polars dask tqdm
!pip install -q -e . --no-deps
# For --use-ilp only:  !pip install -q -c /kaggle/working/constraints.txt pyscipopt

import tracking_cellmot   # verify the package resolves before moving on
print("setup done -- tracking_cellmot importable")

In [ ]:
import os, glob
def _find(kind):
    # Bounded-depth search (depths 0-3). A recursive '**' glob descends into every
    # zarr chunk directory and effectively hangs, so probe explicit levels instead.
    for pat in (f"/kaggle/input/{kind}",
                f"/kaggle/input/*/{kind}",
                f"/kaggle/input/*/*/{kind}",
                f"/kaggle/input/*/*/*/{kind}"):
        for d in glob.glob(pat):
            if os.path.isdir(d) and glob.glob(f"{d}/*.zarr"):
                return d
    return None
TRAIN = _find("train"); TEST = _find("test")
# Export as env vars so "$TRAIN"/"$TEST"/"$CELLMOT_DATA_DIR" expand inside the !python cells.
os.environ["CELLMOT_DATA_DIR"] = TRAIN or ""   # scripts default to this
os.environ["TRAIN"] = TRAIN or ""
os.environ["TEST"] = TEST or ""
print("train:", TRAIN, "|", len(glob.glob(f"{TRAIN}/*.geff")) if TRAIN else 0, "geffs")
print("test :", TEST,  "|", len(glob.glob(f"{TEST}/*.zarr")) if TEST else 0, "zarrs")
assert TRAIN and TEST, "Competition data not found under /kaggle/input -- add it via 'Add Input'."

In [ ]:
import sys
if "/kaggle/working/repo/src" not in sys.path:
    sys.path.insert(0, "/kaggle/working/repo/src")
import os, numpy as np, polars as pl, tracksdata as td
from pathlib import Path
from geff import GeffMetadata
from tracking_cellmot.io import DEFAULT_SCALE, list_datasets, open_dataset

NID, SRC, TGT = (td.DEFAULT_ATTR_KEYS.NODE_ID,
                 td.DEFAULT_ATTR_KEYS.EDGE_SOURCE, td.DEFAULT_ATTR_KEYS.EDGE_TARGET)
data_dir = Path(os.environ["CELLMOT_DATA_DIR"])
names = [p.name for p in list_datasets(data_dir, require_geff=True)]
print(f"{len(names)} datasets in {data_dir}\n")

rows, all_disp = [], []
for name in names:
    g = td.graph.IndexedRXGraph.from_geff(data_dir / f"{name}.geff")
    g = g[0] if isinstance(g, tuple) else g
    nn, ne = g.num_nodes(), g.num_edges()
    try:  est = (GeffMetadata.read(data_dir/f"{name}.geff").extra or {}).get("estimated_number_of_nodes")
    except Exception: est = None
    try:  scale = open_dataset(data_dir/name, load_image=False).scale
    except FileNotFoundError: scale = DEFAULT_SCALE
    ids = g.node_ids()
    na = g.node_attrs(attr_keys=[NID, "t", "z", "y", "x"])
    outd, ind = np.asarray(g.out_degree(ids)), np.asarray(g.in_degree(ids))
    ndiv = int((outd >= 2).sum())
    disp, dt_gap = np.array([]), 0
    if ne:
        ea = g.edge_attrs(attr_keys=[SRC, TGT])
        s = na.rename({NID: SRC, "t": "t_s", "z": "z_s", "y": "y_s", "x": "x_s"})
        t = na.rename({NID: TGT, "t": "t_t", "z": "z_t", "y": "y_t", "x": "x_t"})
        sz, sy, sx = scale
        e = ea.join(s, on=SRC, how="left").join(t, on=TGT, how="left").with_columns(
            (pl.col("t_t") - pl.col("t_s")).alias("dt"),
            (((pl.col("z_t")-pl.col("z_s"))*sz)**2 + ((pl.col("y_t")-pl.col("y_s"))*sy)**2
             + ((pl.col("x_t")-pl.col("x_s"))*sx)**2).sqrt().alias("disp"))
        disp = e["disp"].drop_nulls().to_numpy(); dt_gap = int((e["dt"].to_numpy() > 1).sum())
        all_disp.append(disp)
    rows.append(dict(dataset=name, nodes=nn, edges=ne, frames=int(na["t"].n_unique()),
        divisions=ndiv, div_rate=(ndiv/nn if nn else float("nan")),
        annot_frac=(nn/est if est else float("nan")),
        starts=int((ind==0).sum()), ends=int((outd==0).sum()), dt_gap=dt_gap,
        disp_p50=(float(np.percentile(disp,50)) if disp.size else float("nan")),
        disp_p90=(float(np.percentile(disp,90)) if disp.size else float("nan")),
        disp_max=(float(disp.max()) if disp.size else float("nan"))))

df = pl.DataFrame(rows)
with pl.Config(tbl_rows=200, tbl_cols=20, fmt_str_lengths=40): print(df)
D = np.concatenate([d for d in all_disp if d.size]) if all_disp else np.array([])
total_gap = int(df["dt_gap"].sum())
print(f"\nTOTAL nodes={int(df['nodes'].sum())} edges={int(df['edges'].sum())} "
      f"divisions={int(df['divisions'].sum())} gap_edges(dt>1)={total_gap} | "
      + (f"edge disp um p50/p90/max = {np.percentile(D,50):.2f}/{np.percentile(D,90):.2f}/{D.max():.2f} "
         if D.size else "no edges ") + "(node-match radius = 7 um)")
print("gap-closing verdict:",
      "worth building -- some GT edges skip frames" if total_gap else
      "skip it -- every GT edge is between consecutive frames (dt=1)")
df.write_csv("/kaggle/working/eda.csv")

In [ ]:
# Reproduce the trainer's deterministic seed-0 90/10 split and save it, so training
# and the local validation below score on the SAME held-out videos.
import json, glob, os, random
stems = sorted(os.path.basename(p)[:-5] for p in glob.glob(f"{TRAIN}/*.zarr")
               if os.path.exists(f"{TRAIN}/{os.path.basename(p)[:-5]}.geff"))
random.Random(0).shuffle(stems)
n_val = max(1, len(stems) // 10)
json.dump([{"split": 0, "train": stems[n_val:], "test": stems[:n_val]}],
          open("/kaggle/working/cv_splits.json", "w"))
print(f"{len(stems)} datasets -> {len(stems) - n_val} train / {n_val} val (held out for scoring)")

In [ ]:
# Baseline convergence run (upstream, no PU). --max-iters caps each epoch so the ~4.7 min
# eval overhead is amortized; watch `recall`/`best` each epoch for the plateau
# (the best checkpoint is saved automatically).
!cd /kaggle/working/repo && python scripts/train_unet_transformer.py \
    --method baseline --split 0 --splits /kaggle/working/cv_splits.json \
    --epochs 6 --max-iters 4000 \
    --batch-size 1 --num-workers 4

import os
WEIGHTS = "/kaggle/working/repo/weights/baseline/split_0/edge_predictor_best.pth"
os.environ["WEIGHTS"] = WEIGHTS
assert os.path.exists(WEIGHTS), f"No checkpoint at {WEIGHTS} -- training didn't finish; check the log above."
print("checkpoint OK:", WEIGHTS)

In [ ]:
# Local CV on held-out val videos (--slice :5 for a quick signal; drop it for the full 19-video val).
# Output under method 'baseline_val' so it never mixes with the test predictions.
!cd /kaggle/working/repo && python scripts/predict_unet_transformer.py --method baseline_val --split 0 \
    --data-dir "$CELLMOT_DATA_DIR" --splits /kaggle/working/cv_splits.json \
    --weights "$WEIGHTS" --det-threshold 0.99 --slice :5 --evaluate

In [ ]:
import json, glob, os
test_dir = os.environ["TEST"]
names = sorted(os.path.basename(p)[:-5] for p in glob.glob(f"{test_dir}/*.zarr"))
json.dump([{"split": 0, "train": [], "test": names}], open("/kaggle/working/test_splits.json", "w"))
print(len(names), "test videos")

!cd /kaggle/working/repo && python scripts/predict_unet_transformer.py --method baseline --split 0 \
    --data-dir "$TEST" --splits /kaggle/working/test_splits.json \
    --weights "$WEIGHTS" --det-threshold 0.99

In [ ]:
!cd /kaggle/working/repo && python scripts/geffs_to_csv.py \
    --in-dir predictions/kaggle/baseline/split_0 \
    --csv /kaggle/working/submission.csv
import polars as pl; print(pl.read_csv("/kaggle/working/submission.csv").head())